IMPORTS

In [1]:
from typing import List, Union
from pathlib import Path

import string
import pandas as pd
import numpy as np
import itertools

C:\Users\zscoman\AppData\Local\Temp\ipykernel_23024\3085926645.py:5: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


# ORIGINAL BATCH SUM STAT FUNCTION

In [2]:
def _batch_summary_stats_(csv_path_list: List[str],
                         out_path: str,
                         out_preffix: str):
    """" 
    csv_path_list: List[str],
        A list of path strings where .csv files to analyze are located.
    out_path: str,
        A path string where the summary data file will be output to
    out_preffix: str
        The prefix used to name the output file.    
    """
    ds_count = 0
    fl_count = 0
    ###################
    # Read in the csv files and combine them into one of each type
    ###################
    org_tabs = []
    contact_tabs = []
    dist_tabs = []
    region_tabs = []
    
    org = "_organelles"
    contacts = "_contacts"
    dist = "_distributions"
    regions = "_regions"

    for loc in csv_path_list:
        ds_count = ds_count + 1
        files_store = sorted(loc.glob("*.csv"))
        for file in files_store:
            fl_count = fl_count + 1
            stem = file.stem

            # org = "organelles" TODO: get rid of these (moved to the above)
            # contacts = "contacts"
            # dist = "distributions"
            # regions = "_regions"

            if org in stem:
                test_orgs = pd.read_csv(file, index_col=0)
                test_orgs.insert(0, "dataset", stem[:-11])
                org_tabs.append(test_orgs)
            if contacts in stem:
                test_contact = pd.read_csv(file, index_col=0)
                test_contact.insert(0, "dataset", stem[:-9])
                contact_tabs.append(test_contact)
            if dist in stem:
                test_dist = pd.read_csv(file, index_col=0)
                test_dist.insert(0, "dataset", stem[:-14])
                dist_tabs.append(test_dist)
            if regions in stem:
                test_regions = pd.read_csv(file, index_col=0)
                test_regions.insert(0, "dataset", stem[:-8])
                region_tabs.append(test_regions)
            
    org_df = pd.concat(org_tabs,axis=0, join='outer')
    contacts_df = pd.concat(contact_tabs,axis=0, join='outer')
    dist_df = pd.concat(dist_tabs,axis=0, join='outer')
    regions_df = pd.concat(region_tabs,axis=0, join='outer')

    ###################
    # adding new metrics to the original sheets
    ###################
    # TODO: include these labels when creating the original sheets
    contact_cnt = contacts_df[["dataset", "image_name", "object", "label", "volume"]]
    contact_cnt[["orgA", "orgB"]] = contact_cnt["object"].str.split('X', expand=True)
    contact_cnt[["A_ID", "B_ID"]] = contact_cnt["label"].str.split('_', expand=True)
    contact_cnt["A"] = contact_cnt["orgA"] +"_" + contact_cnt["A_ID"].astype(str)
    contact_cnt["B"] = contact_cnt["orgB"] +"_" + contact_cnt["B_ID"].astype(str)

    contact_cnt_percell = contact_cnt[["dataset", "image_name", "orgA", "A_ID", "object", "volume"]].groupby(["dataset", "image_name", "orgA", "A_ID", "object"]).agg(["count", "sum"])
    contact_cnt_percell.columns = ["_".join(col_name).rstrip('_') for col_name in contact_cnt_percell.columns.to_flat_index()]
    unstacked = contact_cnt_percell.unstack(level='object')
    unstacked.columns = ["_".join(col_name).rstrip('_') for col_name in unstacked.columns.to_flat_index()]
    unstacked = unstacked.reset_index()
    for col in unstacked.columns:
        if col.startswith("volume_count_"):
            newname = col.split("_")[-1] + "_count"
            unstacked.rename(columns={col:newname}, inplace=True)
        if col.startswith("volume_sum_"):
            newname = col.split("_")[-1] + "_volume"
            unstacked.rename(columns={col:newname}, inplace=True)
    unstacked.rename(columns={"orgA":"object", "A_ID":"label"}, inplace=True)
    unstacked.set_index(['dataset', 'image_name', 'object', 'label'])

    contact_percellB = contact_cnt[["dataset", "image_name", "orgB", "B_ID", "object", "volume"]].groupby(["dataset", "image_name", "orgB", "B_ID", "object"]).agg(["count", "sum"])
    contact_percellB.columns = ["_".join(col_name).rstrip('_') for col_name in contact_percellB.columns.to_flat_index()]
    unstackedB = contact_percellB.unstack(level='object')
    unstackedB.columns = ["_".join(col_name).rstrip('_') for col_name in unstackedB.columns.to_flat_index()]
    unstackedB = unstackedB.reset_index()
    for col in unstackedB.columns:
        if col.startswith("volume_count_"):
            newname = col.split("_")[-1] + "_count"
            unstackedB.rename(columns={col:newname}, inplace=True)
        if col.startswith("volume_sum_"):
            newname = col.split("_")[-1] + "_volume"
            unstackedB.rename(columns={col:newname}, inplace=True)
    unstackedB.rename(columns={"orgB":"object", "B_ID":"label"}, inplace=True)
    unstackedB.set_index(['dataset', 'image_name', 'object', 'label'])

    contact_cnt = pd.concat([unstacked, unstackedB], axis=0).sort_index(axis=0)
    contact_cnt = contact_cnt.groupby(['dataset', 'image_name', 'object', 'label']).sum().reset_index()
    contact_cnt['label']=contact_cnt['label'].astype("Int64")

    org_df = pd.merge(org_df, contact_cnt, how='left', on=['dataset', 'image_name', 'object', 'label'], sort=True)
    org_df[contact_cnt.columns] = org_df[contact_cnt.columns].fillna(0)

    ###################
    # summary stat group
    ###################
    group_by = ['dataset', 'image_name', 'object']
    sharedcolumns = ["SA_to_volume_ratio", "equivalent_diameter", "extent", "euler_number", "solidity", "axis_major_length"]
    ag_func_standard = ['mean', 'median', 'std']

    ###################
    # summarize shared measurements between org_df and contacts_df
    ###################
    org_cont_tabs = []
    for tab in [org_df, contacts_df]:
        tab1 = tab[group_by + ['volume']].groupby(group_by).agg(['count', 'sum'] + ag_func_standard)
        tab2 = tab[group_by + ['surface_area']].groupby(group_by).agg(['sum'] + ag_func_standard)
        tab3 = tab[group_by + sharedcolumns].groupby(group_by).agg(ag_func_standard)
        shared_metrics = pd.merge(tab1, tab2, 'outer', on=group_by)
        shared_metrics = pd.merge(shared_metrics, tab3, 'outer', on=group_by)
        org_cont_tabs.append(shared_metrics)

    org_summary = org_cont_tabs[0]
    contact_summary = org_cont_tabs[1]

    ###################
    # group metrics from regions_df similar to the above
    ###################
    regions_summary = regions_df[group_by + ['volume', 'surface_area'] + sharedcolumns].set_index(group_by)

    ###################
    # summarize extra metrics from org_df
    ###################
    columns2 = [col for col in org_df.columns if col.endswith(("_count", "_volume"))]
    contact_counts_summary = org_df[group_by + columns2].groupby(group_by).agg(['sum'] + ag_func_standard)
    org_summary = pd.merge(org_summary, contact_counts_summary, 'outer', on=group_by)#left_on=group_by, right_on=True)

    ###################
    # summarize distribution measurements
    ###################
    # organelle distributions
    hist_dfs = []
    for ind in dist_df.index:
        selection = dist_df.loc[[ind]]
        bins_df = pd.DataFrame()
        wedges_df = pd.DataFrame()
        Z_df = pd.DataFrame()

        bins_df[['bins', 'masks', 'obj']] = selection[['XY_bins', 'XY_mask_vox_cnt_perbin', 'XY_obj_vox_cnt_perbin']]
        wedges_df[['bins', 'masks', 'obj']] = selection[['XY_wedges', 'XY_mask_vox_cnt_perwedge', 'XY_obj_vox_cnt_perwedge']]
        Z_df[['bins', 'masks', 'obj']] = selection[['Z_slices', 'Z_mask_vox_cnt', 'Z_obj_vox_cnt']]

        dfs = [selection[['dataset', 'image_name', 'object']].reset_index()]
        for df, prefix in zip([bins_df, wedges_df, Z_df], ["XY_bins_", "XY_wedges_", "Z_slices_"]):
            single_df = pd.DataFrame(list(zip(df["bins"].values[0][1:-1].split(", "), 
                                            df["obj"].values[0][1:-1].split(", "), 
                                            df["masks"].values[0][1:-1].split(", "))), columns =['bins', 'obj', 'mask']).astype(int)
            
            single_df['mask_fract'] = single_df['mask']/single_df['mask'].max()
            single_df['obj_norm'] = (single_df["obj"]/single_df["mask_fract"]).fillna(0)
            single_df['portion_per_bin'] = (single_df["obj"] / single_df["obj"].sum())*100

            if "Z_" in prefix:
                single_df['bins'] = (single_df["bins"]/max(single_df.bins)*10).apply(np.floor)

            sumstats_df = pd.DataFrame()

            s = single_df['bins'].repeat(single_df['obj_norm'])
            sumstats_df['hist_mean']=[s.mean()]
            sumstats_df['hist_median']=[s.median()]
            if single_df['obj_norm'].sum() != 0: sumstats_df['hist_mode']=[s.mode()[0]]
            else: sumstats_df['hist_mode']=['NaN']
            sumstats_df['hist_min']=[s.min()]
            sumstats_df['hist_max']=[s.max()]
            sumstats_df['hist_range']=[s.max() - s.min()]
            sumstats_df['hist_stdev']=[s.std()]
            sumstats_df['hist_skew']=[s.skew()]
            sumstats_df['hist_kurtosis']=[s.kurtosis()]
            sumstats_df['hist_var']=[s.var()]
            sumstats_df.columns = [prefix+col for col in sumstats_df.columns]
            dfs.append(sumstats_df.reset_index())
        combined_df = pd.concat(dfs, axis=1).drop(columns="index")
        hist_dfs.append(combined_df)
    dist_org_summary = pd.concat(hist_dfs, ignore_index=True)

    # nucleus distribution
    nuc_dist_df = dist_df[["dataset", "image_name", 
                        "XY_bins", "XY_center_vox_cnt_perbin", "XY_mask_vox_cnt_perbin",
                        "XY_wedges", "XY_center_vox_cnt_perwedge", "XY_mask_vox_cnt_perwedge",
                        "Z_slices", "Z_center_vox_cnt", "Z_mask_vox_cnt"]].set_index(["dataset", "image_name"])
    nuc_hist_dfs = []
    for idx in nuc_dist_df.index.unique():
        selection = nuc_dist_df.loc[idx].iloc[[0]].reset_index()
        bins_df = pd.DataFrame()
        wedges_df = pd.DataFrame()
        Z_df = pd.DataFrame()

        bins_df[['bins', 'center', 'masks']] = selection[['XY_bins', 'XY_center_vox_cnt_perbin', 'XY_mask_vox_cnt_perbin']]
        wedges_df[['bins', 'center', 'masks']] = selection[['XY_wedges', 'XY_center_vox_cnt_perwedge', 'XY_mask_vox_cnt_perwedge']]
        Z_df[['bins', 'center', 'masks']] = selection[['Z_slices', 'Z_center_vox_cnt', 'Z_mask_vox_cnt']]

        dfs = [selection[['dataset', 'image_name']]]
        for df, prefix in zip([bins_df, wedges_df, Z_df], ["XY_bins_", "XY_wedges_", "Z_slices_"]):
            single_df = pd.DataFrame(list(zip(df["bins"].values[0][1:-1].split(", "), 
                                            df["masks"].values[0][1:-1].split(", "),
                                            df["center"].values[0][1:-1].split(", "))), columns =['bins', 'mask', 'obj']).astype(int)
            single_df['mask_fract'] = single_df['mask']/single_df['mask'].max()
            single_df['obj_norm'] = (single_df["obj"]/single_df["mask_fract"]).fillna(0)
            single_df['portion_per_bin'] = (single_df["obj"] / single_df["obj"].sum())*100
            if "Z_" in prefix:
                single_df['bins'] = (single_df["bins"]/max(single_df.bins)*10).apply(np.floor)

            sumstats_df = pd.DataFrame()

            s = single_df['bins'].repeat(single_df['obj_norm'])
            sumstats_df['hist_mean']=[s.mean()]
            sumstats_df['hist_median']=[s.median()]
            if single_df['obj_norm'].sum() != 0: sumstats_df['hist_mode']=[s.mode()[0]]
            else: sumstats_df['hist_mode']=['NaN']
            sumstats_df['hist_min']=[s.min()]
            sumstats_df['hist_max']=[s.max()]
            sumstats_df['hist_range']=[s.max() - s.min()]
            sumstats_df['hist_stdev']=[s.std()]
            sumstats_df['hist_skew']=[s.skew()]
            sumstats_df['hist_kurtosis']=[s.kurtosis()]
            sumstats_df['hist_var']=[s.var()]
            sumstats_df.columns = [prefix+col for col in sumstats_df.columns]
            dfs.append(sumstats_df.reset_index())
        combined_df = pd.concat(dfs, axis=1).drop(columns="index")
        nuc_hist_dfs.append(combined_df)
    dist_center_summary = pd.concat(nuc_hist_dfs, ignore_index=True)
    dist_center_summary.insert(2, column="object", value="nuc")

    dist_summary = pd.concat([dist_org_summary, dist_center_summary], axis=0).set_index(group_by).sort_index()

    ###################
    # add normalization
    ###################
    # organelle area fraction
    area_fractions = []
    for idx in org_summary.index.unique():
        org_vol = org_summary.loc[idx][('volume', 'sum')]
        cell_vol = regions_summary.loc[idx[:-1] + ('cell',)]["volume"]
        afrac = org_vol/cell_vol
        area_fractions.append(afrac)
    org_summary[('volume', 'fraction')] = area_fractions
    # TODO: add in line to reorder the level=0 columns here

    # contact sites volume normalized
    norm_toA_list = []
    norm_toB_list = []
    for col in contact_summary.index:
        norm_toA_list.append(contact_summary.loc[col][('volume', 'sum')]/org_summary.loc[col[:-1]+(col[-1].split('X')[0],)][('volume', 'sum')])
        norm_toB_list.append(contact_summary.loc[col][('volume', 'sum')]/org_summary.loc[col[:-1]+(col[-1].split('X')[1],)][('volume', 'sum')])
    contact_summary[('volume', 'norm_to_A')] = norm_toA_list
    contact_summary[('volume', 'norm_to_B')] = norm_toB_list

    # number and area of individuals organelle involved in contact
    cont_cnt = org_df[group_by]
    cont_cnt[[col.split('_')[0] for col in org_df.columns if col.endswith(("_count"))]] = org_df[[col for col in org_df.columns if col.endswith(("_count"))]].astype(bool)
    cont_cnt_perorg = cont_cnt.groupby(group_by).agg('sum')
    cont_cnt_perorg.columns = pd.MultiIndex.from_product([cont_cnt_perorg.columns, ['count_in']])
    for col in cont_cnt_perorg.columns:
        cont_cnt_perorg[(col[0], 'num_fraction_in')] = cont_cnt_perorg[col].values/org_summary[('volume', 'count')].values
    cont_cnt_perorg.sort_index(axis=1, inplace=True)
    org_summary = pd.merge(org_summary, cont_cnt_perorg, on=group_by, how='outer')


    ###################
    # flatten datasheets and combine
    # TODO: restructure this so that all of the datasheets and unstacked and then reorded based on shared level 0 columns before flattening
    ###################
    # org flattening
    org_final = org_summary.unstack(-1)
    for col in org_final.columns:
        if col[1] in ('count_in', 'num_fraction_in') or col[0].endswith(('_count', '_volume')):
            if col[2] not in col[0]:
                org_final.drop(col,axis=1, inplace=True)
    new_col_order = ['dataset', 'image_name', 'object', 'volume', 'surface_area', 'SA_to_volume_ratio', 
                 'equivalent_diameter', 'extent', 'euler_number', 'solidity', 'axis_major_length', 
                 'ERXLD', 'ERXLD_count', 'ERXLD_volume', 'golgiXER', 'golgiXER_count', 'golgiXER_volume', 
                 'golgiXLD', 'golgiXLD_count', 'golgiXLD_volume', 'golgiXperox', 'golgiXperox_count', 'golgiXperox_volume', 
                 'lysoXER', 'lysoXER_count', 'lysoXER_volume', 'lysoXLD', 'lysoXLD_count', 'lysoXLD_volume', 
                 'lysoXgolgi', 'lysoXgolgi_count', 'lysoXgolgi_volume', 'lysoXmito', 'lysoXmito_count', 'lysoXmito_volume', 
                 'lysoXperox', 'lysoXperox_count', 'lysoXperox_volume', 'mitoXER', 'mitoXER_count', 'mitoXER_volume', 
                 'mitoXLD', 'mitoXLD_count', 'mitoXLD_volume', 'mitoXgolgi', 'mitoXgolgi_count', 'mitoXgolgi_volume', 
                 'mitoXperox', 'mitoXperox_count', 'mitoXperox_volume', 'peroxXER', 'peroxXER_count', 'peroxXER_volume', 
                 'peroxXLD', 'peroxXLD_count', 'peroxXLD_volume']
    new_cols = org_final.columns.reindex(new_col_order, level=0)
    org_final = org_final.reindex(columns=new_cols[0])
    org_final.columns = ["_".join((col_name[-1], col_name[1], col_name[0])) for col_name in org_final.columns.to_flat_index()]

    #renaming, filling "NaN" with 0 when needed, and removing ER_std columns
    for col in org_final.columns:
        if '_count_in_' or '_fraction_in_' in col:
            org_final[col] = org_final[col].fillna(0)
        if col.endswith(("_count_volume","_sum_volume", "_mean_volume", "_median_volume")):
            org_final[col] = org_final[col].fillna(0)
        if col.endswith("_count_volume"):
            org_final.rename(columns={col:col.split("_")[0]+"_count"}, inplace=True)
        if col.startswith("ER_std_"):
            org_final.drop(columns=[col], inplace=True)
    org_final = org_final.reset_index()

    # contacts flattened
    contact_final = contact_summary.unstack(-1)
    contact_final.columns = ["_".join((col_name[-1], col_name[1], col_name[0])) for col_name in contact_final.columns.to_flat_index()]

    #renaming and filling "NaN" with 0 when needed
    for col in contact_final.columns:
        if col.endswith(("_count_volume","_sum_volume", "_mean_volume", "_median_volume")):
            contact_final[col] = contact_final[col].fillna(0)
        if col.endswith("_count_volume"):
            contact_final.rename(columns={col:col.split("_")[0]+"_count"}, inplace=True)
    contact_final = contact_final.reset_index()

    # distributions flattened
    dist_final = dist_summary.unstack(-1)
    dist_final.columns = ["_".join((col_name[1], col_name[0])) for col_name in dist_final.columns.to_flat_index()]
    dist_final = dist_final.reset_index()

    # regions flattened & normalization added
    regions_final = regions_summary.unstack(-1)
    regions_final.columns = ["_".join((col_name[1], col_name[0])) for col_name in regions_final.columns.to_flat_index()]
    regions_final['nuc_area_fraction'] = regions_final['nuc_volume'] / regions_final['cell_volume']
    regions_final = regions_final.reset_index()

    # combining them all
    combined = pd.merge(org_final, contact_final, on=["dataset", "image_name"], how="outer")
    combined = pd.merge(combined, dist_final, on=["dataset", "image_name"], how="outer")
    combined = pd.merge(combined, regions_final, on=["dataset", "image_name"], how="outer").set_index(["dataset", "image_name"])
    combined.columns = [col.replace('sum', 'total') for col in combined.columns]

    ###################
    # export summary sheets
    ###################
    org_summary.to_csv(out_path + f"/{out_preffix}per_org_summarystats.csv")
    contact_summary.to_csv(out_path + f"/{out_preffix}per_contact_summarystats.csv")
    dist_summary.to_csv(out_path + f"/{out_preffix}distribution_summarystats.csv")
    regions_summary.to_csv(out_path + f"/{out_preffix}per_region_summarystats.csv")
    combined.to_csv(out_path + f"/{out_preffix}summarystats_combined.csv")

    print(f"Processing of {fl_count} files from {ds_count} dataset(s) is complete.")
    return f"{fl_count} files from {ds_count} dataset(s) were processed"


# Creating New Subfunctions

## FILE READER

In [3]:
def file_reader(csv_path_list, ds_count:int=0, fl_count:int=0):
    ###################
    # Read in the csv files and combine them into one of each type
    ###################
    org_tabs = []
    contact_tabs = []
    dist_tabs = []
    region_tabs = []
    
    org = "_organelles"
    contacts = "_contacts"
    dist = "_distributions"
    regions = "_regions"

    for loc in csv_path_list:
        if isinstance(loc, str): loc = Path(loc)
        ds_count = ds_count + 1
        files_store = sorted(loc.glob("*.csv"))
        for file in files_store:
            fl_count = fl_count + 1
            stem = file.stem

            if org in stem:
                test_orgs = pd.read_csv(file, index_col=0)
                test_orgs.insert(0, "dataset", stem[:-11])
                org_tabs.append(test_orgs)
            if contacts in stem:
                test_contact = pd.read_csv(file, index_col=0)
                test_contact.insert(0, "dataset", stem[:-9])
                contact_tabs.append(test_contact)
            if dist in stem:
                test_dist = pd.read_csv(file, index_col=0)
                test_dist.insert(0, "dataset", stem[:-14])
                dist_tabs.append(test_dist)
            if regions in stem:
                test_regions = pd.read_csv(file, index_col=0)
                test_regions.insert(0, "dataset", stem[:-8])
                region_tabs.append(test_regions)
            
    org_df = pd.concat(org_tabs,axis=0, join='outer')
    contacts_df = pd.concat(contact_tabs,axis=0, join='outer')
    dist_df = pd.concat(dist_tabs,axis=0, join='outer')
    regions_df = pd.concat(region_tabs,axis=0, join='outer')
    return org_df, contacts_df, dist_df, regions_df, ds_count, fl_count


## Orgs

In [ ]:
def org_summarize(org_df,
                  group_by:list[str]=['dataset', 'image_name', 'object'], 
                  shared_columns:list[str]=["SA_to_volume_ratio", "equivalent_diameter", "extent", "euler_number", "solidity", "axis_major_length"],
                  ag_func_standard: list[str]=['mean', 'median', 'std'],
                  zeros=None):
    
    for col in ['cell', 'subcellular_region']:
        tab1 = org_df[group_by + [col,'volume']].groupby(group_by+[col]).agg(['count', 'sum'] + ag_func_standard)
        tab2 = org_df[group_by + [col,'surface_area']].groupby(group_by+[col]).agg(['sum'] + ag_func_standard)
        tab3 = org_df[group_by + [col]+shared_columns].groupby(group_by+[col]).agg(ag_func_standard)
        shared_metrics = pd.merge(tab1, tab2, 'outer', on=group_by+[col])
        shared_metrics = pd.merge(shared_metrics, tab3, 'outer', on=group_by+[col]).unstack(col)
        if col == 'subcellular_region':
            org_summary = pd.merge(org_summary, shared_metrics, 'inner', on=group_by).swaplevel(i=0, j=-1, axis=1).swaplevel(i=1, j=2, axis=1)
        elif col == 'cell':
            org_summary = shared_metrics
    org_summary = org_summary.reindex(org_summary.columns.get_level_values(0).unique(), level=0, axis=1)
    for fn in ag_func_standard:
        for region in org_summary.columns.get_level_values(0).unique():
            count_one = [count!=1 for count in org_summary[region]["volume"]["count"]]
            org_summary.loc[count_one, ([region],["equivalent_diameter"],[fn])] = zeros
    return org_summary

In [5]:
def summarize_org(org_df, group_by, ag_func_standard):
    columns2 = [col for col in org_df.columns if col.endswith(("_count", "_volume"))]
    contact_counts_summary = org_df[group_by + columns2].groupby(group_by).agg(['sum'] + ag_func_standard)
    org_summary = pd.merge(org_summary, contact_counts_summary, 'outer', on=group_by)#left_on=group_by, right_on=True)
    return org_summary

In [6]:
def org_area_fraction(org_summary, regions_summary):
    for region in org_summary.columns.get_level_values(0).unique():
        area_fraction=[]
        for idx in org_summary.index.unique():
            org_vol = org_summary.loc[idx][(region, 'volume', 'sum')]
            cell_vol = sum([regions_summary.loc[idx[:-1] + (reg.split('-')[0],)]["volume"]["sum"] for reg in region.split(':')[-1].split("_")])
            print(f"{region}: {cell_vol}")
            afrac = org_vol/cell_vol
            area_fraction.append(afrac)
        org_summary[(region, 'volume', 'fraction')] = area_fraction
    return org_summary

In [7]:
def normalize_org(org_df, org_summary, group_by):
    cont_cnt_subcell = org_df[group_by+['subcellular_region']]
    cont_cnt_subcell[[col.split('_')[0] for col in org_df.columns if col.endswith(("_count"))]] = org_df[[col for col in org_df.columns if col.endswith(("_count"))]].astype(bool)
    cont_cnt_perorg_subcell = cont_cnt_subcell.groupby(group_by+['subcellular_region']).agg('sum')
    cont_cnt_perorg_subcell.columns = pd.MultiIndex.from_product([cont_cnt_perorg_subcell.columns, ['count_in']])
    cont_cnt_perorg_subcell = cont_cnt_perorg_subcell.unstack('subcellular_region').swaplevel(i=0, j=-1, axis=1).swaplevel(i=1, j=2, axis=1)
    
    cont_cnt_cell = org_df[group_by+['cell']]
    cont_cnt_cell[[col.split('_')[0] for col in org_df.columns if col.endswith(("_count"))]] = org_df[[col for col in org_df.columns if col.endswith(("_count"))]].astype(bool)
    cont_cnt_perorg_cell = cont_cnt_cell.groupby(group_by+['cell']).agg('sum')
    cont_cnt_perorg_cell.columns = pd.MultiIndex.from_product([cont_cnt_perorg_cell.columns, ['count_in']])
    cont_cnt_perorg_cell = cont_cnt_perorg_cell.unstack('cell').swaplevel(i=0, j=-1, axis=1).swaplevel(i=1, j=2, axis=1)
    cont_cnt_perorg_perreg = pd.merge(cont_cnt_perorg_cell, cont_cnt_perorg_subcell, on=group_by)
    cont_cnt_perorg_perreg = cont_cnt_perorg_perreg.reindex(cont_cnt_perorg_perreg.columns.get_level_values(0).unique(), level=0, axis=1)
    
    for region in org_summary.columns.get_level_values(0).unique():
        for col in cont_cnt_perorg_perreg.columns:
            cont_cnt_perorg_perreg[(region, col[1], 'num_fraction_in')] = cont_cnt_perorg_perreg[col].values/org_summary[(region, 'volume', 'count')].values
    cont_cnt_perorg_perreg.sort_index(axis=1, inplace=True)
    org_summary = pd.merge(org_summary, cont_cnt_perorg_perreg, on=group_by, how='outer')
    return org_summary

In [8]:
def org_flattening(org_summary, org_df, splitter):
    all_combos = []
    all_orgs = list(set(org_df.loc[:, 'object'].tolist()))
    org_final = org_summary.unstack(-1)
    for col in org_final.columns:
        if col[1] in ('count_in', 'num_fraction_in') or col[0].endswith(('_count', '_volume')):
            if col[2] not in col[0]:
                org_final.drop(col,axis=1, inplace=True)
    new_col_order = ['dataset', 'image_name', 'object', 'volume', 'surface_area', 'SA_to_volume_ratio', 
                     'equivalent_diameter', 'extent', 'euler_number', 'solidity', 'axis_major_length'] 
    for n in list(map(lambda x:x+2, (range(len(all_orgs)-1)))):
        all_combos+=itertools.combinations(all_orgs, n)
    combos = [splitter.join(interact) for interact in all_combos]
    for combo in combos:
        new_col_order += [f"{combo}", f"{combo}_count", f"{combo}_volume"]
    new_cols = org_final.columns.reindex(new_col_order, level=0)
    org_final = org_final.reindex(columns=new_cols[0])
    org_final.columns = ["_".join((col_name[-1], col_name[1], col_name[0])) for col_name in org_final.columns.to_flat_index()]
    org_final = org_final.groupby(level=0, axis=1).sum()
    
    #renaming, filling "NaN" with 0 when needed, and removing ER_std columns
    for col in org_final.columns:
        if '_count_in_' or '_fraction_in_' in col:
            org_final[col] = org_final[col].fillna(0)

        if col.endswith(("_count_volume","_sum_volume", "_mean_volume", "_median_volume")):
            org_final[col] = org_final[col].fillna(0)
            
        if col.endswith("_count_volume"):
            org_final.rename(columns={col:col.split("_")[0]+"_count"}, inplace=True)

        if col.startswith("ER_std_"):
            org_final.drop(columns=[col], inplace=True)
            
    org_final = org_final.reset_index()
    return org_final

## Interactions

In [9]:
# TAKES IN:
#       INTERACTION DATA TABLE   -   For reorganizing the data
#       SPLITTER                 -   For determining orgs involved
def batch_summary_interactions(interaction_df, splitter):
    
    # CREATE A NEW DATA TABLE FOR INTERACTION COUNTS
    interaction_cnt = interaction_df[["dataset", "image_name", "object", "label", "volume"]]
    
    # CREATES NEW COLUMNS EQUAL TO MAX NUMBER OF ORGANELLES INVOLVED IN A CONTACT
    # CREATES A NEW COLUMN FOR STORING THE ORGANELLE ID FOR EACH ORGANELLE INVOLVED IN A CONTACT
    # FOR EXAMPLE: mitoXER of 06_01 would become the following
    #              A_ID | orgA | B_ID | orgB
    #               06  | mito |  01  |  ER 
    interaction_cnt[[f"org{cha}" for cha in string.ascii_uppercase[:(len(max(interaction_cnt["object"].str.split(splitter), key=len)))]]] = interaction_cnt["object"].str.split(splitter, expand=True)
    interaction_cnt[[f"{cha}_ID" for cha in string.ascii_uppercase[:(len(max(interaction_cnt["label"].str.split('_'), key=len)))]]] = interaction_cnt["label"].str.split('_', expand=True)
    #iterating from a to val
    unstacked_interactions = []
    for cha in string.ascii_uppercase[:len(max(interaction_cnt["object"].str.split(splitter), key=len))]:
        # DETERMINE WHICH VALUES IN interaction_cnt HAVE THE DESIRED ALPHABETICAL ORGANELLE COUNT
        # i.e. if a contact has 4 organelles in it, the maximum alphabetical organelle count will be "D"
        valid = (interaction_cnt[f"org{cha}"] != None) & (interaction_cnt[f"{cha}_ID"] != None)

        # CREATES A NEW COLUMN WITH THE VALUE OF THE CURRENT ALPHABETICAL CHARACTER
        interaction_cnt[f"{cha}"] = None

        # Note: USED LATER WITH SEPARATING ORG AND INTERACTION DATA
        interaction_cnt.loc[valid, f"{cha}"] = interaction_cnt[f"org{cha}"] + "_" + interaction_cnt[f"{cha}_ID"]

        # CREATES A NEW DATAFRAME FOR PER CELL DATA FOCUSING ONLY ON CURRENT ALPHABETICAL ORGANELLE CHARACTER GROUPED BY CELL
        interaction_cnt_percell = interaction_cnt[["dataset", "image_name", f"org{cha}", f"{cha}_ID", "object", "volume"]].groupby(["dataset", "image_name", f"org{cha}", f"{cha}_ID", "object"]).agg(["count", "sum"])
        interaction_cnt_percell.columns = ["_".join(col_name).rstrip('_') for col_name in interaction_cnt_percell.columns.to_flat_index()]
        unstacked = interaction_cnt_percell.unstack(level='object')
        unstacked.columns = ["_".join(col_name).rstrip('_') for col_name in unstacked.columns.to_flat_index()]
        unstacked = unstacked.reset_index()
        for col in unstacked.columns:
            if col.startswith("volume_count_"):
                newname = col.split("_")[-1] + "_count"
                unstacked.rename(columns={col:newname}, inplace=True)
            if col.startswith("volume_sum_"):
                newname = col.split("_")[-1] + "_volume"
                unstacked.rename(columns={col:newname}, inplace=True)
        unstacked.rename(columns={f"org{cha}":"object", f"{cha}_ID":"label"}, inplace=True)
        unstacked.set_index(['dataset', 'image_name', 'object', 'label'])    
        unstacked_interactions.append(unstacked)
    interaction_cnt = pd.concat(unstacked_interactions, axis=0).sort_index(axis=0)
    interaction_cnt = interaction_cnt.groupby(['dataset', 'image_name', 'object', 'label']).sum().reset_index()                 #adds together all duplicates at the index, then resets the index
    interaction_cnt['label'] = interaction_cnt['label'].astype("Int64")  
    return interaction_cnt

In [10]:
def normalize_interaction_volumes(interaction_summary, region_summary, splitter: str="X"):
    for region in interaction_summary.columns.get_level_values(0).unique():
        norm_to_list = {}
        for idx,cha in enumerate(string.ascii_uppercase[:len(max(interaction_summary.index.get_level_values('object').str.split(splitter), key=len))]):
            for row in interaction_summary.index:
                if cha not in norm_to_list:
                    norm_to_list[cha]=[]
                if ((idx+1) <= len(row[-1].split(splitter))): # continue if nth order 
                    org = row[-1].split(splitter)[idx]
                    if (interaction_summary.loc[row][(region,'volume', 'sum')]>=0) and (region_summary.loc[row[:-1]+(region.split('-')[0],)][(f"{org}_volume", 'sum')] >= 0):
                        norm_to_list[cha].append(interaction_summary.loc[row][(region,'volume', 'sum')]/region_summary.loc[row[:-1]+(region.split('-')[0],)][(f"{org}_volume", 'sum')])
                    else:
                        norm_to_list[cha].append(None)
                else: # specified interaction is below nth order, leave cell as none
                    norm_to_list[cha].append(None)
        for cha in string.ascii_uppercase[:len(max(interaction_summary.index.get_level_values('object').str.split(splitter), key=len))]:
            interaction_summary[(region, 'volume', f'norm_to_{cha}')] = norm_to_list[cha]
    interaction_summary = interaction_summary.reindex(interaction_summary.columns.get_level_values(0).unique(), level=0, axis=1)
    return interaction_summary

In [11]:
def interaction_summarize(interaction_df,
                        group_by:list[str]=['dataset', 'image_name', 'object'], 
                            shared_columns:list[str]=["SA_to_volume_ratio", "equivalent_diameter", "extent", "euler_number", "solidity", "axis_major_length"],
                            ag_func_standard: list[str]=['mean', 'median', 'std']):

    for col in ['cell', 'subcellular_region']:
        tab1 = interaction_df[group_by + [col,'volume']].groupby(group_by+[col]).agg(['count', 'sum'] + ag_func_standard)
        tab2 = interaction_df[group_by + [col,'surface_area']].groupby(group_by+[col]).agg(['sum'] + ag_func_standard)
        tab3 = interaction_df[group_by + [col]+shared_columns].groupby(group_by+[col]).agg(ag_func_standard)
        shared_metrics = pd.merge(tab1, tab2, 'outer', on=group_by+[col])
        shared_metrics = pd.merge(shared_metrics, tab3, 'outer', on=group_by+[col]).unstack(col)
        if col == 'subcellular_region':
            interaction_summary = pd.merge(interaction_summary, shared_metrics, 'inner', on=group_by).swaplevel(i=0, j=-1, axis=1).swaplevel(i=1, j=2, axis=1)
        elif col == 'cell':
            interaction_summary = shared_metrics
    interaction_summary = interaction_summary.reindex(interaction_summary.columns.get_level_values(0).unique(), level=0, axis=1)
     
    return interaction_summary 

In [12]:
def interaction_flattening(interaction_summary):
    # contacts flattened
    interaction_final = interaction_summary.unstack(-1)
    interaction_final.columns = ["_".join((col_name[-1], col_name[1], col_name[0])) for col_name in interaction_final.columns.to_flat_index()]
    interaction_final = interaction_final.groupby(level=0, axis=1).sum()
    #renaming and filling "NaN" with 0 when needed
    for col in interaction_final.columns:
        if col.endswith(("_count_volume","_sum_volume", "_mean_volume", "_median_volume")):
            interaction_final[col] = interaction_final[col].fillna(0)
        if col.endswith("_count_volume"):
            interaction_final.rename(columns={col:col.split("_")[0]+"_count"}, inplace=True)
    interaction_final = interaction_final.reset_index()
    return interaction_final

## Region

In [13]:
def summarize_regions(regions_df, 
                      org_list: list[str],
                      group_by:list[str]=['dataset', 'image_name', 'object'], 
                      shared_columns:list[str]=["SA_to_volume_ratio", "equivalent_diameter", "extent", "euler_number", "solidity", "axis_major_length"],
                      ag_func_standard: list[str]=['mean', 'median', 'std'],
                      zeros=None):
    tab1 = regions_df[group_by + ['volume']].groupby(group_by).agg(['count', 'sum'] + ag_func_standard)
    tab2 = regions_df[group_by + ['surface_area'] + [f'{org}_volume' for org in org_list]].groupby(group_by).agg(['sum'] + ag_func_standard)
    tab3 = regions_df[group_by + shared_columns].groupby(group_by).agg(ag_func_standard)
    shared_metrics = pd.merge(tab1, tab2, 'outer', on=group_by)
    regions_summary = pd.merge(shared_metrics, tab3, 'outer', on=group_by)
    count_one = [count==1 for count in regions_summary["volume"]["count"]] # List of bools for each row having 1 as its count or not
    regions_summary.loc[count_one, (["volume"],["mean"])] = zeros
    regions_summary.loc[count_one, (["surface_area"],["mean"])] = zeros
    for col in shared_columns+["volume", "surface_area"]:
        regions_summary.loc[count_one, ([col],["median"])] = zeros
        regions_summary.loc[count_one, ([col],["std"])] = zeros

    return regions_summary

In [14]:
def finalize_regions(regions_summary):
    regions_final = regions_summary.unstack(-1)
    regions_final.columns = ["_".join((col_name[-1], col_name[1], col_name[0])) for col_name in regions_final.columns.to_flat_index()]
    regions_final['nuc_area_fraction'] = regions_final['nuc_mean_volume'] / (regions_final['cell_mean_volume'])
    regions_final = regions_final.reset_index()
    for col in regions_final.columns:
        if col.endswith(("_count_volume","_sum_volume", "_mean_volume", "_median_volume")):
            regions_final[col] = regions_final[col].fillna(0)
        if col.endswith("_count_volume"):
            regions_final.rename(columns={col:col.split("_")[0]+"_count"}, inplace=True)
    regions_final = regions_final.reset_index()
    return regions_final

In [15]:
def zeros_singles(input_df: pd.DataFrame, shared_columns: list[str], zeros = None):
    for region in input_df.columns.get_level_values(0).unique():
        count_one = [count==1 for count in input_df[region]["volume"]["count"]] # List of bools for each row having 1 as its count or not
        input_df.loc[count_one, ([region],["volume"],["mean"],)] = zeros
        input_df.loc[count_one, ([region],["surface_area"],["mean"])] = zeros
        for col in shared_columns+["volume", "surface_area"]:
            input_df.loc[count_one, ([region],[col],["median"])] = zeros
            input_df.loc[count_one, ([region],[col],["std"])] = zeros
    return input_df

In [16]:
def subregion_subunit_combinder(in_df):
    def boarder_type_combinder(x):
        return x.split(':')[0]+':'+"_".join(list(set([i.split('-')[0] for i in x.split(':')[1].split('_')])))
    out_df = in_df.copy()
    is_boarder = out_df['subcellular_region'].str.split(':').str[0] == 'boarder'
    out_df.loc[is_boarder, 'subcellular_region'] = out_df.loc[is_boarder, 'subcellular_region'].apply(lambda x: boarder_type_combinder(x))
    out_df['subcellular_region'] = out_df['subcellular_region'].str.split('-').str[0]
    return out_df

# New OVERALL Function

In [ ]:
def batch_summary_stats(csv_path_list: List[str],
                         out_path: str,
                         out_preffix: str,
                         splitter:str="X"):
    """" 
    csv_path_list: List[str],
        A list of path strings where .csv files to analyze are located.
    out_path: str,
        A path string where the summary data file will be output to
    out_preffix: str
        The prefix used to name the output file.    
    """
    ds_count = 0
    fl_count = 0
    ###################
    # Read in the csv files and combine them into one of each type
    ###################
    org_df, interaction_df, dist_df, regions_df, ds_count, fl_count = file_reader(csv_path_list=csv_path_list)

    org_df = subregion_subunit_combinder(org_df)
    interaction_df = subregion_subunit_combinder(interaction_df)

    ###################
    # adding new metrics to the original sheets
    ###################
    # TODO: include these labels when creating the original sheets
    interaction_cnt = batch_summary_interactions(interaction_df=interaction_df, splitter=splitter)
    org_df = pd.merge(org_df, interaction_cnt, how='left', on=['dataset', 'image_name', 'object', 'label'], sort=True)
    org_df[interaction_cnt.columns] = org_df[interaction_cnt.columns].fillna(0)
    
    ###################
    # summary stat group
    ###################
    group_by = ['dataset', 'image_name', 'object']
    sharedcolumns = ["SA_to_volume_ratio", "equivalent_diameter", "extent", "euler_number", "solidity", "axis_major_length"]
    ag_func_standard = ['mean', 'median', 'std']

    ###################
    # summarize shared measurements between org_df and contacts_df
    ###################
    org_summary = org_summarize(org_df, group_by=group_by, shared_columns=sharedcolumns, ag_func_standard=ag_func_standard)
    interaction_summary = interaction_summarize(interaction_df, group_by=group_by, shared_columns=sharedcolumns, ag_func_standard=ag_func_standard)
    
    ###################
    # group metrics from regions_df similar to the above
    ###################
    regions_summary = summarize_regions(regions_df=regions_df, org_list=org_df.object.unique(), group_by=group_by, shared_columns=sharedcolumns)
    
    # set single metrics to zeros (no median or standard deviation)
    org_summary = zeros_singles(org_summary, sharedcolumns)
    interaction_summary = zeros_singles(interaction_summary, sharedcolumns)
    #regions_summary = zeros_singles(regions_summary, sharedcolumns)
   

    columns2 = [col for col in org_df.columns if col.endswith(("_count", "_volume"))]
    contact_counts_summary_subcell = org_df[group_by + ['subcellular_region'] + columns2].groupby(group_by+['subcellular_region']).agg(['sum'] + ag_func_standard).unstack('subcellular_region').swaplevel(i=0, j=-1, axis=1).swaplevel(i=1, j=2, axis=1)
    contact_counts_summary_cell = org_df[group_by + ['cell'] + columns2].groupby(group_by+['cell']).agg(['sum'] + ag_func_standard).unstack('cell').swaplevel(i=0, j=-1, axis=1).swaplevel(i=1, j=2, axis=1)
    org_summary = pd.merge(org_summary, contact_counts_summary_subcell, 'outer', on=group_by)#left_on=group_by, right_on=True)
    org_summary = pd.merge(org_summary, contact_counts_summary_cell, 'outer', on=group_by)#left_on=group_by, right_on=True)
    org_summary = org_summary.reindex(org_summary.columns.get_level_values(0).unique(), level=0, axis=1)
    

    ###################
    # summarize distribution measurements
    ###################
    # organelle distributions
    hist_dfs = []
    for ind in dist_df.index:
        selection = dist_df.loc[[ind]]
        bins_df = pd.DataFrame()
        wedges_df = pd.DataFrame()
        Z_df = pd.DataFrame()

        bins_df[['bins', 'masks', 'obj']] = selection[['XY_bins', 'XY_mask_vox_cnt_perbin', 'XY_obj_vox_cnt_perbin']]
        wedges_df[['bins', 'masks', 'obj']] = selection[['XY_wedges', 'XY_mask_vox_cnt_perwedge', 'XY_obj_vox_cnt_perwedge']]
        Z_df[['bins', 'masks', 'obj']] = selection[['Z_slices', 'Z_mask_vox_cnt', 'Z_obj_vox_cnt']]

        dfs = [selection[['dataset', 'image_name', 'object']].reset_index()]
        for df, prefix in zip([bins_df, wedges_df, Z_df], ["XY_bins_", "XY_wedges_", "Z_slices_"]):
            single_df = pd.DataFrame(list(zip(df["bins"].values[0][1:-1].split(", "), 
                                            df["obj"].values[0][1:-1].split(", "), 
                                            df["masks"].values[0][1:-1].split(", "))), columns =['bins', 'obj', 'mask']).astype(int)
            
            single_df['mask_fract'] = single_df['mask']/single_df['mask'].max()
            single_df['obj_norm'] = (single_df["obj"]/single_df["mask_fract"]).fillna(0)
            single_df['portion_per_bin'] = (single_df["obj"] / single_df["obj"].sum())*100

            if "Z_" in prefix:
                single_df['bins'] = (single_df["bins"]/max(single_df.bins)*10).apply(np.floor)

            sumstats_df = pd.DataFrame()

            s = single_df['bins'].repeat(single_df['obj_norm'])
            sumstats_df['hist_mean']=[s.mean()]
            sumstats_df['hist_median']=[s.median()]
            if single_df['obj_norm'].sum() != 0: sumstats_df['hist_mode']=[s.mode()[0]]
            else: sumstats_df['hist_mode']=['NaN']
            sumstats_df['hist_min']=[s.min()]
            sumstats_df['hist_max']=[s.max()]
            sumstats_df['hist_range']=[s.max() - s.min()]
            sumstats_df['hist_stdev']=[s.std()]
            sumstats_df['hist_skew']=[s.skew()]
            sumstats_df['hist_kurtosis']=[s.kurtosis()]
            sumstats_df['hist_var']=[s.var()]
            sumstats_df.columns = [prefix+col for col in sumstats_df.columns]
            dfs.append(sumstats_df.reset_index())
        combined_df = pd.concat(dfs, axis=1).drop(columns="index")
        hist_dfs.append(combined_df)
    dist_org_summary = pd.concat(hist_dfs, ignore_index=True)

    # nucleus distribution
    nuc_dist_df = dist_df[["dataset", "image_name", 
                        "XY_bins", "XY_center_vox_cnt_perbin", "XY_mask_vox_cnt_perbin",
                        "XY_wedges", "XY_center_vox_cnt_perwedge", "XY_mask_vox_cnt_perwedge",
                        "Z_slices", "Z_center_vox_cnt", "Z_mask_vox_cnt"]].set_index(["dataset", "image_name"])
    nuc_hist_dfs = []
    for idx in nuc_dist_df.index.unique():
        selection = nuc_dist_df.loc[idx].iloc[[0]].reset_index()
        bins_df = pd.DataFrame()
        wedges_df = pd.DataFrame()
        Z_df = pd.DataFrame()

        bins_df[['bins', 'center', 'masks']] = selection[['XY_bins', 'XY_center_vox_cnt_perbin', 'XY_mask_vox_cnt_perbin']]
        wedges_df[['bins', 'center', 'masks']] = selection[['XY_wedges', 'XY_center_vox_cnt_perwedge', 'XY_mask_vox_cnt_perwedge']]
        Z_df[['bins', 'center', 'masks']] = selection[['Z_slices', 'Z_center_vox_cnt', 'Z_mask_vox_cnt']]

        dfs = [selection[['dataset', 'image_name']]]
        for df, prefix in zip([bins_df, wedges_df, Z_df], ["XY_bins_", "XY_wedges_", "Z_slices_"]):
            single_df = pd.DataFrame(list(zip(df["bins"].values[0][1:-1].split(", "), 
                                            df["masks"].values[0][1:-1].split(", "),
                                            df["center"].values[0][1:-1].split(", "))), columns =['bins', 'mask', 'obj']).astype(int)
            single_df['mask_fract'] = single_df['mask']/single_df['mask'].max()
            single_df['obj_norm'] = (single_df["obj"]/single_df["mask_fract"]).fillna(0)
            single_df['portion_per_bin'] = (single_df["obj"] / single_df["obj"].sum())*100
            if "Z_" in prefix:
                single_df['bins'] = (single_df["bins"]/max(single_df.bins)*10).apply(np.floor)

            sumstats_df = pd.DataFrame()

            s = single_df['bins'].repeat(single_df['obj_norm'])
            sumstats_df['hist_mean']=[s.mean()]
            sumstats_df['hist_median']=[s.median()]
            if single_df['obj_norm'].sum() != 0: sumstats_df['hist_mode']=[s.mode()[0]]
            else: sumstats_df['hist_mode']=['NaN']
            sumstats_df['hist_min']=[s.min()]
            sumstats_df['hist_max']=[s.max()]
            sumstats_df['hist_range']=[s.max() - s.min()]
            sumstats_df['hist_stdev']=[s.std()]
            sumstats_df['hist_skew']=[s.skew()]
            sumstats_df['hist_kurtosis']=[s.kurtosis()]
            sumstats_df['hist_var']=[s.var()]
            sumstats_df.columns = [prefix+col for col in sumstats_df.columns]
            dfs.append(sumstats_df.reset_index())
        combined_df = pd.concat(dfs, axis=1).drop(columns="index")
        nuc_hist_dfs.append(combined_df)
    dist_center_summary = pd.concat(nuc_hist_dfs, ignore_index=True)
    dist_center_summary.insert(2, column="object", value="nuc")

    dist_summary = pd.concat([dist_org_summary, dist_center_summary], axis=0).set_index(group_by).sort_index()

    ###################
    # add normalization
    ###################
    # organelle area fraction
    org_summary = org_area_fraction(org_summary, regions_summary)
    # TODO: add in line to reorder the level=0 columns here

    # contact sites volume normalized
    display(regions_summary)
    interaction_summary = normalize_interaction_volumes(interaction_summary, regions_summary, splitter)

    # number and area of individuals organelle involved in contact
    org_summary = normalize_org(org_df, org_summary, group_by)


    ###################
    # flatten datasheets and combine
    # TODO: restructure this so that all of the datasheets and unstacked and then reorded based on shared level 0 columns before flattening
    ###################
    # org flattening
    org_final = org_flattening(org_summary, org_df, splitter)

    interaction_final = interaction_flattening(interaction_summary)

    # distributions flattened
    dist_final = dist_summary.unstack(-1)
    dist_final.columns = ["_".join((col_name[1], col_name[0])) for col_name in dist_final.columns.to_flat_index()]
    dist_final = dist_final.reset_index()

    # regions flattened & normalization added
    regions_final=finalize_regions(regions_summary)

    # combining them all
    combined = pd.merge(org_final, interaction_final, on=["dataset", "image_name"], how="outer")
    combined = pd.merge(combined, dist_final, on=["dataset", "image_name"], how="outer")
    combined = pd.merge(combined, regions_final, on=["dataset", "image_name"], how="outer").set_index(["dataset", "image_name"])
    combined.columns = [col.replace('sum', 'total') for col in combined.columns]

    ###################
    # export summary sheets
    ###################
    org_summary = org_summary.reindex(org_summary.columns.get_level_values(0).unique(), level=0, axis=1)
    org_summary = org_summary.reindex(org_summary.columns.get_level_values(1).unique(), level=1, axis=1)
    org_summary.to_csv(out_path + f"/{out_preffix}per_org_summarystats.csv")
    interaction_summary = interaction_summary.reindex(interaction_summary.columns.get_level_values(0).unique(), level=0, axis=1)
    interaction_summary = interaction_summary.reindex(interaction_summary.columns.get_level_values(1).unique(), level=1, axis=1)
    interaction_summary.to_csv(out_path + f"/{out_preffix}per_contact_summarystats.csv")
    dist_summary.to_csv(out_path + f"/{out_preffix}distribution_summarystats.csv")
    regions_summary.to_csv(out_path + f"/{out_preffix}per_region_summarystats.csv")
    combined.to_csv(out_path + f"/{out_preffix}summarystats_combined.csv")

    print(f"Processing of {fl_count} files from {ds_count} dataset(s) is complete.")
    return f"{fl_count} files from {ds_count} dataset(s) were processed"

In [18]:
batch_summary_stats([Path("C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/neurites/outputs/neurite_soma")],
                    "C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/neurites/outputs/sumstat",
                    out_preffix="neurite_soma_sumstat")

C:\Users\zscoman\AppData\Local\Temp\ipykernel_23024\4013592713.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  interaction_cnt[[f"org{cha}" for cha in string.ascii_uppercase[:(len(max(interaction_cnt["object"].str.split(splitter), key=len)))]]] = interaction_cnt["object"].str.split(splitter, expand=True)
C:\Users\zscoman\AppData\Local\Temp\ipykernel_23024\4013592713.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  interaction_cnt[[f"org{cha}" for cha in string.ascii_uppercase[:(len(max(interaction_

cell-1  \
                                                                                       volume   
                                                                                        count   
dataset                      image_name                                         object          
neurite_checks_neurites_soma 01252024_MSi08L_iN_Day7_BR5_N01_Unmixing_0_cmle... ER          1   
                                                                                LD          6   
                                                                                golgi      91   
                                                                                lyso      178   
                                                                                mito      112   
                                                                                perox     113   

                                                                                                    \
                                                                                                     
                                                                                               sum   
dataset                      image_name                                         object               
neurite_checks_neurites_soma 01252024_MSi08L_iN_Day7_BR5_N01_Unmixing_0_cmle... ER      215.249579   
                                                                                LD        0.222672   
                                                                                golgi     9.417450   
                                                                                lyso     23.722438   
                                                                                mito    101.830819   
                                                                                perox     5.391361   

                                                                                                    \
                                                                                                     
                                                                                              mean   
dataset                      image_name                                         object               
neurite_checks_neurites_soma 01252024_MSi08L_iN_Day7_BR5_N01_Unmixing_0_cmle... ER      215.249579   
                                                                                LD        0.037112   
                                                                                golgi     0.103488   
                                                                                lyso      0.133272   
                                                                                mito      0.909204   
                                                                                perox     0.047711   

                                                                                                    \
                                                                                                     
                                                                                            median   
dataset                      image_name                                         object               
neurite_checks_neurites_soma 01252024_MSi08L_iN_Day7_BR5_N01_Unmixing_0_cmle... ER      215.249579   
                                                                                LD        0.023617   
                                                                                golgi     0.076473   
                                                                                lyso      0.065227   
                                                                                mito      0.504948   
                                                                                perox     0.042735   

                                                     

UnboundLocalError: local variable 'interaction_summary' referenced before assignment